In [ ]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc
from dash import html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from animal_crud import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name

username = "aacuser"
password = "SNHU1234"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)

# Rescue dog filters
RESCUE_FILTERS = {
    "Water Rescue": { 
        "breeds": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"],
        "sex": ["Intact Female"],
        "age": (26, 156),
    },
    "Mountain or Wilderness Rescue": {
        "breeds": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"],
        "sex": ["Intact Male"],
        "age": (26, 156),
    },
    "Disaster or Individual Tracking": {
        "breeds": ["Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound", "Rottweiler"],
        "sex": ["Intact Male"],
        "age": (20, 300),
    },
}

def build_query(filter_type: str) -> dict:
    if not filter_type or filter_type =="ALL":
        return {}
    
    spec = RESCUE_FILTERS.get(filter_type, {})
    breeds = spec.get("breeds", [])
    sexes  = spec.get("sex", [])
    age_lo, age_hi = spec.get("age", (None, None))

    q = {"animal_type": {"$in": ["Dog", "dog"]}}
    if breeds:
        q["breed"] = {"$in": breeds}
    if sexes:
        q["sex_upon_outcome"] = {"$in": sexes}
    if age_lo is not None and age_hi is not None:
        q["age_upon_outcome_in_weeks"] = {"$gte": age_lo, "$lte": age_hi}
    return q

#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#FIX ME Add in Grazioso Salvare’s logo
image_filename = 'GraziosoSalvareLogo.png'  # update if your filename differs
encoded_image = None
if os.path.exists(image_filename):
    with open(image_filename, 'rb') as f:
        encoded_image = base64.b64encode(f.read()).decode()

#FIX ME Place the HTML image tag in the line below into the app.layout code according to your design
#FIX ME Also remember to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))


app.layout = html.Div([
#    html.Div(id='hidden-div', style={'display':'none'}),
    html.Center(html.B(html.H1('Greg Gordon\'s CS-340 Dashboard'))),
    html.Hr(),

        
     # Logo area (optional if file missing)
    html.Div([
        html.Img(src=f'data:image/png;base64,{encoded_image}', style={'height': '64px'}) if encoded_image
        else html.Div("(Add GraziosoSalvareLogo.png to show logo)")
    ], style={'textAlign': 'center'}),
#FIXME Add in code for the interactive filtering options. For example, Radio buttons, drop down, checkboxes, etc.


html.Hr(),
html.Div(
    children=[
        # --- Interactive filtering options (minimal) ---
        html.Label("Rescue Type Filter (enforces breeds + sex + age from spec)"),
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Reset (All)  ', 'value': 'ALL'},
                {'label': 'Water Rescue  ', 'value': 'Water Rescue'},
                {'label': 'Mountain or Wilderness Rescue  ', 'value': 'Mountain or Wilderness Rescue'},
                {'label': 'Disaster or Individual Tracking  ', 'value': 'Disaster or Individual Tracking'},
            ],
            value='ALL',
            inline=True
        ),
    ]
),


    html.Hr(),
    dash_table.DataTable(id='datatable-id',
                         columns=[
                            {"name": "name", "id": "name"},
                            {"name": "breed", "id": "breed"},
                            {"name": "sex_upon_outcome", "id": "sex_upon_outcome"},
                            {"name": "age_upon_outcome", "id": "age_upon_outcome"},
                            {"name": "outcome_type", "id": "outcome_type"},
                            {"name": "location_lat", "id": "location_lat"},
                            {"name": "location_long", "id": "location_long"},
    ],                         
                         data=df.to_dict('records'),
                        page_size=10,
                         sort_action="native",
                         filter_action="native",
                         column_selectable="single",
                         row_selectable="single",
                         selected_rows=[0],
                         style_table={'overflowX': 'auto'}
                        ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################



    
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
    query = build_query(filter_type)
    data = db.read(query) or []

    for d in data:
        d.pop('_id', None)

    return data

# Went with a bar graph due to better scaling with filter changes.
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    dff = pd.DataFrame(viewData) if viewData else df.copy()
    if dff.empty or 'breed' not in dff.columns:
        return [html.Div("No data to display.")]

    counts = (
        dff['breed']
        .value_counts(dropna=True)
        .reset_index()
        .rename(columns={'index': 'breed', 'breed': 'count'})
    )

    # keep the top 12, group the rest into "Other"
    top_n = 12
    if len(counts) > top_n:
        other_sum = counts['count'][top_n:].sum()
        counts = counts.head(top_n)
        counts.loc[len(counts)] = ['Other', other_sum]

    fig = px.bar(
        counts.sort_values('count'),
        x='count', y='breed',
        orientation='h',
        title='Top Breeds in Current View'
    )
    fig.update_layout(
        margin=dict(l=10, r=10, t=40, b=40),
        height=400 + 20 * len(counts),  # auto-grow a bit with categories
        xaxis_title='Count', yaxis_title=''
    )
    return [dcc.Graph(figure=fig)]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    selected_columns = selected_columns or []  # <-- important
    return [{
        'if': { 'column_id': col },
        'background_color': '#D2F3FF'
    } for col in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, selected_rows):
    if not viewData:
        # Nothing to show yet
        return [html.Div("No rows in view.")]

    dff = pd.DataFrame(viewData)

    # Default to first row if nothing selected
    row_idx = (selected_rows or [0])[0]
    row_idx = max(0, min(row_idx, len(dff) - 1))

    # Safely pull fields by column name
    lat = dff.at[row_idx, 'location_lat'] if 'location_lat' in dff.columns else None
    lon = dff.at[row_idx, 'location_long'] if 'location_long' in dff.columns else None
    breed = dff.at[row_idx, 'breed'] if 'breed' in dff.columns else "Unknown"
    name  = dff.at[row_idx, 'name'] if 'name' in dff.columns else "Unknown"

    # If lat/lon missing, center on Austin
    center_lat, center_lon = (30.75, -97.48)
    if pd.notna(lat) and pd.notna(lon):
        center_lat, center_lon = float(lat), float(lon)

    marker_children = []
    if pd.notna(lat) and pd.notna(lon):
        marker_children = [
            dl.Marker(position=[float(lat), float(lon)], children=[
                dl.Tooltip(str(breed)),
                dl.Popup([html.H1("Animal Name"), html.P(str(name))])
            ])
        ]

    return [
        dl.Map(
            style={'width': '100%', 'height': '500px'},
            center=[center_lat, center_lon],
            zoom=10,
            children=[dl.TileLayer(id="base-layer-id")] + marker_children
        )
    ]

app.run_server(debug=True)


Dash app running on http://127.0.0.1:27010/
